Ignoring Pyspark WARN: Unable to load native-hadoop library for your platform... using built-in Java classes where applicable

ERROR (likely some version compatibility issue between spark, scala, kafka in the included JAR):
`An error occurred while calling o50.load. java.lang.NoClassDefFoundError: Could not initialize class org.apache.spark.sql.kafka010.KafkaSourceProvider$`

- Spark Master UI → http://localhost:8080
- Spark Worker UI → http://localhost:8081 (haven't tried it yet)

In [1]:
!pyspark --version        # or alternatively (same output): spark-submit --version

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/05 15:59:20 WARN Utils: Your hostname, sohang-VivoBook-ASUS-Laptop-X510UFO, resolves to a loopback address: 127.0.1.1; using 192.168.1.31 instead (on interface wlp2s0)
26/07/05 15:59:20 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Welcome to
      ____              __
     / __/__  ___ _____/ /__
    _\ \/ _ \/ _ `/ __/  '_/
   /___/ .__/\_,_/_/ /_/\_\   version 4.1.1
      /_/
                        
Using Scala version 2.13.17, OpenJDK 64-Bit Server VM, 17.0.19
Branch HEAD
Compiled by user runner on 2026-01-02T11:55:02Z
Revision c0690c763bafabd08e7079d1137fa0a769a05bae
Url https://github.com/apache/spark
Type --help for more information.


In [2]:
!~/kafka_2.13-4.3.0/bin/kafka-topics.sh --version

4.3.0


In [3]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import DoubleType, LongType, StringType, StructField, StructType
from pyspark.sql.window import Window


In [ ]:
session = (
    SparkSession.builder.appName("SensorStreamingPipeline")
    .config("spark.sql.streaming.schemaInference", "true")
    .config("spark.sql.streaming.forceDeleteTempCheckpointLocation", "true")
    .config(
        # FINALLY THIS JAR WORKS!!!
        "spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.2"
    )
    .getOrCreate()
)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/05 15:59:25 WARN Utils: Your hostname, sohang-VivoBook-ASUS-Laptop-X510UFO, resolves to a loopback address: 127.0.1.1; using 192.168.1.31 instead (on interface wlp2s0)
26/07/05 15:59:25 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/sohang/.local/bin/miniconda3/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/sohang/.ivy2.5.2/cache
The jars for the packages stored in: /home/sohang/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-73374aba-42a9-4e16-9d85-b957befe2c6d;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.1.2 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.1.2 in central
	found org.apache.kafka#kaf

In [5]:
schema = StructType(
    [
        StructField("sensor_id", StringType(), False),
        StructField("temperature", DoubleType(), True),
        StructField("timestamp", StringType(), False),
        StructField("status", StringType(), False),
    ]
)


In [6]:
kafka_site = "localhost:9092"
topic = "sensor_DA25M622"
df = (
    session.readStream.format("kafka")
    .option("kafka.bootstrap.servers", kafka_site)
    .option("subscribe", topic)
    .option("startingOffsets", "earliest")
    .load()
)


In [7]:
df

DataFrame[key: binary, value: binary, topic: string, partition: int, offset: bigint, timestamp: timestamp, timestampType: int]